In [5]:

import pandas as pd

df = pd.read_parquet('../data/second/validated_rides_2023_01.parquet')


In [6]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

In [7]:
df.head(10)

,pickup_datetime,user_location_id
0,2023-01-01 00:32:10,161
1,2023-01-01 00:55:08,43
2,2023-01-01 00:25:04,48
3,2023-01-01 00:03:48,138
4,2023-01-01 00:10:29,107
5,2023-01-01 00:50:34,161
6,2023-01-01 00:09:22,239
7,2023-01-01 00:27:12,142
8,2023-01-01 00:21:44,164
9,2023-01-01 00:39:42,141


In [8]:
df['pickup_hour'] = df['pickup_datetime'].dt.floor('H')

ValueError: Invalid frequency: H. Failed to parse with error message: ValueError("Invalid frequency: H. Failed to parse with error message: KeyError('H'). Did you mean h?") Did you mean h?

In [9]:
df.dtypes

pickup_datetime     datetime64[us]
user_location_id             int64
dtype: object

In [10]:
df['pickup_hour'] = df['pickup_datetime'].dt.floor('h')

In [11]:
df

,pickup_datetime,user_location_id,pickup_hour
0,2023-01-01 00:32:10,161,2023-01-01 00:00:00
1,2023-01-01 00:55:08,43,2023-01-01 00:00:00
2,2023-01-01 00:25:04,48,2023-01-01 00:00:00
3,2023-01-01 00:03:48,138,2023-01-01 00:00:00
4,2023-01-01 00:10:29,107,2023-01-01 00:00:00
...,...,...,...
3066761,2023-01-31 23:58:34,107,2023-01-31 23:00:00
3066762,2023-01-31 23:31:09,112,2023-01-31 23:00:00
3066763,2023-01-31 23:01:05,114,2023-01-31 23:00:00
3066764,2023-01-31 23:40:00,230,2023-01-31 23:00:00


In [21]:
new_df = df.groupby(["pickup_hour","user_location_id"]).size().reset_index()

In [22]:
new_df

,pickup_hour,user_location_id,0
0,2023-01-01 00:00:00,4,19
1,2023-01-01 00:00:00,7,3
2,2023-01-01 00:00:00,12,1
3,2023-01-01 00:00:00,13,14
4,2023-01-01 00:00:00,24,20
...,...,...,...
71486,2023-01-31 23:00:00,261,5
71487,2023-01-31 23:00:00,262,11
71488,2023-01-31 23:00:00,263,41
71489,2023-01-31 23:00:00,264,40


In [23]:
new_df.rename(columns={0: 'rides'}, inplace=True)
new_df

,pickup_hour,user_location_id,rides
0,2023-01-01 00:00:00,4,19
1,2023-01-01 00:00:00,7,3
2,2023-01-01 00:00:00,12,1
3,2023-01-01 00:00:00,13,14
4,2023-01-01 00:00:00,24,20
...,...,...,...
71486,2023-01-31 23:00:00,261,5
71487,2023-01-31 23:00:00,262,11
71488,2023-01-31 23:00:00,263,41
71489,2023-01-31 23:00:00,264,40


In [27]:
!pip install tqdm

In [28]:
from tqdm import tqdm

ModuleNotFoundError: No module named 'tqdm'

In [29]:
import sys
!{sys.executable} -m pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [31]:
from tqdm import tqdm


In [32]:
for i in tqdm(range(1000)):
    pass

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 2610021.16it/s]


In [35]:
def add_missing_slots(df):
    import pandas as pd
    from tqdm import tqdm
    
    location_ids = df['user_location_id'].unique()
    
    full_range = pd.date_range(
        df['pickup_hour'].min(),
        df['pickup_hour'].max(),
        freq='h'   # ✅ FIXED
    )
    
    output = pd.DataFrame()
    
    for location_id in tqdm(location_ids):
        
        temp = df.loc[
            df.user_location_id == location_id,
            ['pickup_hour', 'rides']
        ]
        
        temp = temp.set_index('pickup_hour')
        
        temp = temp.reindex(full_range, fill_value=0)
        
        temp['user_location_id'] = location_id
        
        output = pd.concat([output, temp])
    
    output = output.reset_index().rename(columns={'index': 'pickup_hour'})
    
    return output

In [36]:
final_df = add_missing_slots(new_df)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 257/257 [00:00<00:00, 359.52it/s]
